# Setup

In [ ]:
import os
import numpy as np
import pandas as pd

import plotly.express as px

In [ ]:
DOWNLOADED_SUMMARY = '/mnt/craig/pan_phylon/S_pyogenes/metadata/filtered_downloaded_species_summary.csv'
DOWNLOADED_METADATA = '/mnt/craig/pan_phylon/S_pyogenes/metadata/filtered_downloaded_species_metadata.csv'

In [ ]:
downloaded_summary = pd.read_csv(DOWNLOADED_SUMMARY, index_col=0, dtype='object')
downloaded_metadata = pd.read_csv(DOWNLOADED_METADATA, index_col=0, dtype='object')

display(
    downloaded_summary.shape,
    downloaded_summary.head()
)

# De-duplicate entries

## Ensure `biosample_accession` is unique & drop duplicates

In [ ]:
downloaded_metadata = downloaded_metadata.drop_duplicates(subset=['biosample_accession'])

display(
    downloaded_metadata.shape,
    downloaded_metadata.head()
)

## (Optional) Go through and remove any empty or mostly empty columns

In [ ]:
# mostly_empty_cols = pd.DataFrame(index=downloaded_metadata.columns, columns=['num_NA'])

# for col in downloaded_metadata.columns:
#     mostly_empty_cols.at[col, 'num_NA'] = downloaded_metadata[col].isna().sum()


# # Drop anything for which over 50% of the metadata entries are N/A
# px.histogram(mostly_empty_cols, x='num_NA', nbins=100)

In [ ]:
# cutoff_cols_NA =  downloaded_metadata.shape[0]/2
# cond = mostly_empty_cols.num_NA < cutoff_cols_NA

# mostly_full_cols = mostly_empty_cols[cond]
# mostly_full_cols.loc['plasmids', 'num_NA'] = mostly_empty_cols.loc['plasmids', 'num_NA']

# mostly_full_cols

In [ ]:
#scrubbed_metadata = downloaded_metadata[mostly_full_cols.index]
scrubbed_metadata = downloaded_metadata.copy()

display(
    scrubbed_metadata.shape,
    scrubbed_metadata.head()
)

In [ ]:
scrubbed_summary = downloaded_summary.copy()

common_genomes = sorted(set(scrubbed_metadata['genome_id']).intersection(set(scrubbed_summary['genome_id'])))
scrubbed_summary = scrubbed_summary.set_index('genome_id').loc[common_genomes].reset_index()

display(
    scrubbed_summary.shape,
    scrubbed_summary.head()
)

# Save cleaned summary & metadata files

In [ ]:
filepath = DOWNLOADED_METADATA.split('filtered_downloaded_species_metadata')[0]
filepath = os.path.join(filepath, 'scrubbed_species_metadata.csv')

filepath

In [ ]:
scrubbed_metadata.to_csv(filepath)

In [ ]:
filepath = DOWNLOADED_SUMMARY.split('filtered_downloaded_species_summary')[0]
filepath = os.path.join(filepath, 'scrubbed_species_summary.csv')

filepath

In [ ]:
scrubbed_summary.to_csv(filepath)

In [ ]:
downloaded_summary.shape[0] - scrubbed_summary.shape[0] # Number of genomes filtered during de-duplication

In [ ]:
# Ensure reference strain still exists after scrubbing
display(
    scrubbed_metadata.set_index('genome_id').loc['1314.739'],
    scrubbed_summary.set_index('genome_id').loc['1314.739']
)

In [ ]:
scrubbed_summary.checkm_completeness.min()

In [ ]:
scrubbed_summary.checkm_completeness.isna().sum()

In [ ]:
scrubbed_summary.checkm_contamination.max()

In [ ]:
scrubbed_summary.checkm_contamination.isna().sum()

In [ ]:
scrubbed_summary.checkm_contamination.astype('float').hist()

In [ ]:
pd.read_csv('/mnt/craig/pan_phylon/S_pyogenes/raw/checkm_output/checkm_qa_filtered.tsv', sep='\t').columns